# 11 - Ingest Crawled Documentation with Gemini and Pinecone

## Objective

Continue from notebook 10 and reproduce the instructor's documentation-ingestion logic using Gemini embeddings and Pinecone.

```text
Saved Tavily crawl artifact
        ↓
Recreate LangChain Document objects
        ↓
RecursiveCharacterTextSplitter
chunk_size = 4000
chunk_overlap = 200
        ↓
Validate and enrich chunk metadata
        ↓
Create deterministic record IDs
        ↓
Gemini gemini-embedding-001
1,536 values per chunk
        ↓
Controlled batch upsert
        ↓
Pinecone index: langchain-docs-helper-gemini
namespace: langchain-docs
```

## Instructor logic preserved

- `RecursiveCharacterTextSplitter`
- `chunk_size=4000`
- `chunk_overlap=200`
- Batched vector-store indexing
- Batch-level error handling
- Success and failure counts
- Pipeline logging and summary
- Source URL metadata

## Intentional adaptations

- Gemini replaces OpenAI embeddings.
- Pinecone is the active vector store.
- Deterministic IDs make reruns idempotent at the record level.
- Controlled sequential batching is the default to respect Gemini API limits.
- An optional controlled-async implementation is included for learning, but disabled.
- Crawling is not repeated. The notebook loads the JSONL artifact created by notebook 10.

> Rerunning an upsert with the same IDs prevents duplicate Pinecone records, but Gemini still regenerates embeddings and consumes API quota.


## 1. Optional package installation

Run this cell only if an import fails, then restart the kernel.


In [ ]:
# %pip install -U \
#     langchain-core \
#     langchain-google-genai \
#     langchain-pinecone \
#     langchain-text-splitters \
#     pinecone \
#     python-dotenv \
#     truststore


## 2. Check the active Python kernel

The executable should point to the `nlp_env` Conda environment.


In [1]:
import sys


print("Python executable:", sys.executable)
print("Python version:", sys.version)


Python executable: C:\Users\inkk1r04\AppData\Local\anaconda3\envs\nlp_env\python.exe
Python version: 3.10.20 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:42:35) [MSC v.1942 64 bit (AMD64)]


## 3. Configure corporate SSL

Run this before importing the external API clients.


In [2]:
import truststore


truststore.inject_into_ssl()


print("Windows trust store injected into Python SSL.")


Windows trust store injected into Python SSL.


## 4. Import ingestion components

In [3]:
import asyncio
import hashlib
import json
import math
import os
import re
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pinecone import Pinecone


print("Imports completed successfully.")


Imports completed successfully.


## 5. Notebook-friendly logging helpers

These preserve the role of the instructor's `logger.py` while keeping this notebook self-contained.


In [4]:
class Colors:
    """ANSI color codes for terminals that support colored output."""

    PURPLE = "\033[95m"
    CYAN = "\033[96m"
    DARKCYAN = "\033[36m"
    GREEN = "\033[92m"
    YELLOW = "\033[93m"
    RED = "\033[91m"
    BOLD = "\033[1m"
    END = "\033[0m"


def log_info(message, color=Colors.CYAN):
    print(f"{color}INFO: {message}{Colors.END}")


def log_success(message):
    print(f"{Colors.GREEN}SUCCESS: {message}{Colors.END}")


def log_warning(message):
    print(f"{Colors.YELLOW}WARNING: {message}{Colors.END}")


def log_error(message):
    print(f"{Colors.RED}ERROR: {message}{Colors.END}")


def log_header(message):
    print()
    print(Colors.BOLD + Colors.PURPLE + "=" * 72 + Colors.END)
    print(Colors.BOLD + Colors.PURPLE + message + Colors.END)
    print(Colors.BOLD + Colors.PURPLE + "=" * 72 + Colors.END)
    print()


log_success("Logging helpers are ready.")


SUCCESS: Logging helpers are ready.


## 6. Locate and load `.env`

Expected variables:

```env
GOOGLE_API_KEY=your-google-key
PINECONE_API_KEY=your-pinecone-key
DOCUMENTATION_INDEX_NAME=langchain-docs-helper-gemini
```

The Tavily key is not required here because this notebook does not crawl the web.


In [5]:
CURRENT_DIR = Path.cwd().resolve()


env_candidates = [
    CURRENT_DIR / ".env",
    CURRENT_DIR.parent / "langchain-course" / ".env",
]


ENV_PATH = next(
    (
        candidate
        for candidate in env_candidates
        if candidate.exists()
    ),
    None,
)


if ENV_PATH is None:
    raise FileNotFoundError(
        "Could not find .env in the current notebook folder "
        "or the sibling langchain-course folder."
    )


load_dotenv(dotenv_path=ENV_PATH)


required_variables = [
    "GOOGLE_API_KEY",
    "PINECONE_API_KEY",
    "DOCUMENTATION_INDEX_NAME",
]


missing_variables = [
    variable_name
    for variable_name in required_variables
    if not os.getenv(variable_name)
]


if missing_variables:
    raise ValueError(
        "Missing environment variables: "
        + ", ".join(missing_variables)
    )


print("Environment file:", ENV_PATH)

for variable_name in required_variables:
    print(
        f"{variable_name} loaded:",
        bool(os.getenv(variable_name)),
    )


Environment file: C:\Users\inkk1r04\OneDrive - Kellogg Company\Desktop\Work\0. DS Capability\Courses\Langchain\langchain-course\.env
GOOGLE_API_KEY loaded: True
PINECONE_API_KEY loaded: True
DOCUMENTATION_INDEX_NAME loaded: True


## 7. Define shared ingestion configuration

These settings must remain consistent in the later retrieval backend.


In [26]:
INDEX_NAME = os.environ["DOCUMENTATION_INDEX_NAME"]
NAMESPACE = "langchain-docs"

EMBEDDING_MODEL = "gemini-embedding-001"
EMBEDDING_DIMENSION = 1536

CHUNK_SIZE = 4000
CHUNK_OVERLAP = 200

# The instructor demonstrates batches. A conservative default is used here
# because the Gemini integration may issue one provider call per document.
BATCH_SIZE = 10
SECONDS_BETWEEN_BATCHES = 60

ARTIFACT_DIR = CURRENT_DIR / "documentation_crawl_artifacts"
DOCUMENTS_PATH = ARTIFACT_DIR / "langchain_docs_documents.jsonl"
CRAWL_SUMMARY_PATH = ARTIFACT_DIR / "langchain_docs_crawl_summary.json"
INGESTION_MANIFEST_PATH = (
    ARTIFACT_DIR / "langchain_docs_ingestion_manifest.json"
)


print("Index name:", INDEX_NAME)
print("Namespace:", NAMESPACE)
print("Embedding model:", EMBEDDING_MODEL)
print("Embedding dimension:", EMBEDDING_DIMENSION)
print("Chunk size:", CHUNK_SIZE)
print("Chunk overlap:", CHUNK_OVERLAP)
print("Batch size:", BATCH_SIZE)
print("Prepared document artifact:", DOCUMENTS_PATH)


Index name: langchain-docs-helper-gemini
Namespace: langchain-docs
Embedding model: gemini-embedding-001
Embedding dimension: 1536
Chunk size: 4000
Chunk overlap: 200
Batch size: 10
Prepared document artifact: C:\Users\inkk1r04\OneDrive - Kellogg Company\Desktop\Work\0. DS Capability\Courses\Langchain\langchain-course-jupyter\documentation_crawl_artifacts\langchain_docs_documents.jsonl


## 8. Confirm notebook 10 artifacts exist

The prepared JSONL file is required. The crawl summary is optional but useful for lineage.


In [27]:
if not DOCUMENTS_PATH.exists():
    raise FileNotFoundError(
        f"Prepared documentation artifact not found: {DOCUMENTS_PATH}\n"
        "Run notebook 10 and save the validated documents first."
    )


print("Prepared document artifact exists:", DOCUMENTS_PATH.exists())
print("Artifact size in bytes:", DOCUMENTS_PATH.stat().st_size)
print("Crawl summary exists:", CRAWL_SUMMARY_PATH.exists())


if CRAWL_SUMMARY_PATH.exists():
    crawl_summary = json.loads(
        CRAWL_SUMMARY_PATH.read_text(
            encoding="utf-8"
        )
    )

    print("Crawl summary:")

    for key, value in crawl_summary.items():
        print(f"- {key}: {value}")
else:
    crawl_summary = {}


Prepared document artifact exists: True
Artifact size in bytes: 511602
Crawl summary exists: True
Crawl summary:
- start_url: https://docs.langchain.com/oss/python/langchain/overview
- allowed_domain: docs.langchain.com
- crawl_timestamp_utc: 2026-09-28T11:24:53.940458+00:00
- raw_result_count: 41
- accepted_url_count: 37
- rejected_result_count: 4
- deduplicated_document_count: 37
- total_characters: 473727
- minimum_page_characters: 1675
- maximum_page_characters: 61537
- average_page_characters: 12803.4


## 9. Load the saved pages as LangChain `Document` objects

This cell performs local file I/O only. It makes no Tavily, Gemini, or Pinecone calls.


In [28]:
source_documents = []


with DOCUMENTS_PATH.open(
    "r",
    encoding="utf-8",
) as input_file:
    for line_number, line in enumerate(
        input_file,
        start=1,
    ):
        if not line.strip():
            continue

        row = json.loads(line)

        page_content = str(
            row.get("page_content", "")
        ).strip()

        metadata = row.get("metadata", {})

        if not page_content:
            log_warning(
                f"Skipping empty page at JSONL line {line_number}."
            )
            continue

        if not isinstance(metadata, dict):
            raise TypeError(
                f"Metadata at JSONL line {line_number} is not a dictionary."
            )

        source_documents.append(
            Document(
                page_content=page_content,
                metadata=metadata,
            )
        )


if not source_documents:
    raise ValueError(
        "No usable source documents were loaded from the JSONL artifact."
    )


log_success(
    f"Loaded {len(source_documents)} source documents."
)


print("First source metadata:")
print(source_documents[0].metadata)
print()
print("First source preview:")

print(source_documents[0].page_content[:1200])


SUCCESS: Loaded 37 source documents.
First source metadata:
{'source': 'https://docs.langchain.com/oss/python/langchain/overview', 'source_domain': 'docs.langchain.com', 'source_path': '/oss/python/langchain/overview', 'crawl_page_number': 0, 'crawl_timestamp_utc': '2026-09-28T11:24:53.940458+00:00', 'crawler': 'tavily', 'content_format': 'markdown', 'page_content_hash': 'b7721a4893bc5070440abdd6b25e93bdfe5de1c505e698f4dfa8587546a5ad28', 'alternate_sources': []}

First source preview:
## Documentation Index

Fetch the complete documentation index at: </llms.txt>

Use this file to discover all available pages before exploring further.

Fall AMA Series: Live sessions on building, evaluating, deploying, and continously improving agents with LangSmith. [Register now →](https://events.langchain.com/fall-product-series/)

![light logo](https://mintcdn.com/langchain-5e9cc07a/nQm-sjd_MByLhgeW/images/brand/langchain-docs-dark-blue.png?fit=max&auto=format&n=nQm-sjd_MByLhgeW&q=85&s=5babf1a1962208

## 10. Validate source-document metadata

Only public documentation URLs and Pinecone-compatible metadata values should proceed to chunking.

Pinecone metadata is kept simple:

- strings
- numbers
- booleans
- lists of strings

Complex nested dictionaries are not sent.


In [29]:
ALLOWED_METADATA_SCALAR_TYPES = (
    str,
    int,
    float,
    bool,
)


def is_supported_metadata_value(value):
    """Return True for metadata values supported by this pipeline."""

    if isinstance(value, ALLOWED_METADATA_SCALAR_TYPES):
        return True

    if isinstance(value, list):
        return all(
            isinstance(item, str)
            for item in value
        )

    return False


metadata_issues = []


for document_number, document in enumerate(
    source_documents
):
    source = str(
        document.metadata.get("source", "")
    )

    if not source.startswith("https://"):
        metadata_issues.append({
            "document_number": document_number,
            "reason": "invalid_source_url",
            "source": source,
        })

    unsupported_keys = [
        key
        for key, value in document.metadata.items()
        if not is_supported_metadata_value(value)
    ]

    if unsupported_keys:
        metadata_issues.append({
            "document_number": document_number,
            "reason": "unsupported_metadata_values",
            "keys": unsupported_keys,
        })


if metadata_issues:
    raise ValueError(
        f"Source metadata validation failed: {metadata_issues[:10]}"
    )


log_success("Source-document metadata validation passed.")


SUCCESS: Source-document metadata validation passed.


## 11. Create the instructor-aligned recursive text splitter

The instructor uses:

```python
RecursiveCharacterTextSplitter(
    chunk_size=4000,
    chunk_overlap=200,
)
```

The recursive splitter tries progressively smaller boundaries so that paragraphs, lines, and words are preserved where possible before falling back to character-level splitting.


In [30]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
)


print("Text splitter:", type(text_splitter).__name__)
print("Chunk size:", CHUNK_SIZE)
print("Chunk overlap:", CHUNK_OVERLAP)


Text splitter: RecursiveCharacterTextSplitter
Chunk size: 4000
Chunk overlap: 200


## 12. Split source pages into embedding chunks

This cell is local and makes no external API calls.


In [31]:
log_header("DOCUMENT CHUNKING PHASE")
log_info(
    f"Processing {len(source_documents)} source documents "
    f"with chunk size {CHUNK_SIZE} and overlap {CHUNK_OVERLAP}."
)


chunks = text_splitter.split_documents(
    source_documents
)


if not chunks:
    raise ValueError("The text splitter produced no chunks.")


log_success(
    f"Created {len(chunks)} chunks from "
    f"{len(source_documents)} source documents."
)



DOCUMENT CHUNKING PHASE

INFO: Processing 37 source documents with chunk size 4000 and overlap 200.
SUCCESS: Created 150 chunks from 37 source documents.


## 13. Inspect chunk-size statistics

Chunk sizes should be close to, but not always exactly, 4,000 characters because the splitter respects natural text boundaries.


In [32]:
chunk_lengths = [
    len(chunk.page_content)
    for chunk in chunks
]


print("Chunk count:", len(chunks))
print("Shortest chunk:", min(chunk_lengths))
print("Longest chunk:", max(chunk_lengths))
print(
    "Average chunk length:",
    round(sum(chunk_lengths) / len(chunk_lengths), 1),
)
print(
    "Chunks above configured size:",
    sum(length > CHUNK_SIZE for length in chunk_lengths),
)


length_bands = Counter(
    "under_500"
    if length < 500
    else "500_to_1999"
    if length < 2000
    else "2000_to_3999"
    if length < 4000
    else "4000_or_more"
    for length in chunk_lengths
)


print("Chunk-size bands:", dict(length_bands))


Chunk count: 150
Shortest chunk: 253
Longest chunk: 3997
Average chunk length: 3218.0
Chunks above configured size: 0
Chunk-size bands: {'2000_to_3999': 124, 'under_500': 4, '500_to_1999': 22}


## 14. Inspect a few chunks before embedding

This is the last visual quality check before spending Gemini quota.


In [33]:
PREVIEW_CHUNKS = 3
PREVIEW_CHARACTERS = 1800


for preview_number, chunk in enumerate(
    chunks[:PREVIEW_CHUNKS],
    start=1,
):
    print("=" * 100)
    print("Preview number:", preview_number)
    print("Length:", len(chunk.page_content))
    print("Source:", chunk.metadata.get("source"))
    print("Metadata:", chunk.metadata)
    print("Content preview:")
    print(chunk.page_content[:PREVIEW_CHARACTERS])
    print()


Preview number: 1
Length: 3959
Source: https://docs.langchain.com/oss/python/langchain/overview
Metadata: {'source': 'https://docs.langchain.com/oss/python/langchain/overview', 'source_domain': 'docs.langchain.com', 'source_path': '/oss/python/langchain/overview', 'crawl_page_number': 0, 'crawl_timestamp_utc': '2026-09-28T11:24:53.940458+00:00', 'crawler': 'tavily', 'content_format': 'markdown', 'page_content_hash': 'b7721a4893bc5070440abdd6b25e93bdfe5de1c505e698f4dfa8587546a5ad28', 'alternate_sources': []}
Content preview:
## Documentation Index

Fetch the complete documentation index at: </llms.txt>

Use this file to discover all available pages before exploring further.

Fall AMA Series: Live sessions on building, evaluating, deploying, and continously improving agents with LangSmith. [Register now →](https://events.langchain.com/fall-product-series/)

![light logo](https://mintcdn.com/langchain-5e9cc07a/nQm-sjd_MByLhgeW/images/brand/langchain-docs-dark-blue.png?fit=max&auto=format&

## 15. Enrich chunk metadata and create deterministic IDs

Each record ID is derived from:

```text
normalized source URL
+ source-relative chunk number
+ chunk text
```

If the same source page and chunk text are ingested again, the ID remains the same and Pinecone upserts the existing record instead of creating a duplicate.


In [34]:
chunk_ids = []
source_chunk_counters = Counter()
current_ingestion_timestamp = datetime.now(
    timezone.utc
).isoformat()


for global_chunk_number, chunk in enumerate(chunks):
    source_url = str(
        chunk.metadata.get("source", "")
    )

    source_chunk_number = source_chunk_counters[
        source_url
    ]

    source_chunk_counters[source_url] += 1

    # Build a new allowlisted metadata dictionary instead of carrying
    # every loader field into the external vector database.
    sanitized_metadata = {
        "source": source_url,
        "source_domain": str(
            chunk.metadata.get("source_domain", "")
        ),
        "source_path": str(
            chunk.metadata.get("source_path", "")
        ),
        "crawler": str(
            chunk.metadata.get("crawler", "tavily")
        ),
        "content_format": str(
            chunk.metadata.get("content_format", "markdown")
        ),
        "page_content_hash": str(
            chunk.metadata.get("page_content_hash", "")
        ),
        "global_chunk_number": global_chunk_number,
        "source_chunk_number": source_chunk_number,
        "chunk_character_count": len(chunk.page_content),
        "chunk_size_setting": CHUNK_SIZE,
        "chunk_overlap_setting": CHUNK_OVERLAP,
        "embedding_model": EMBEDDING_MODEL,
        "embedding_dimension": EMBEDDING_DIMENSION,
        "ingestion_timestamp_utc": current_ingestion_timestamp,
    }

    alternate_sources = chunk.metadata.get(
        "alternate_sources",
        [],
    )

    if alternate_sources:
        sanitized_metadata["alternate_sources"] = [
            str(source)
            for source in alternate_sources
        ]

    chunk.metadata = sanitized_metadata

    id_source = (
        f"{source_url}|"
        f"{source_chunk_number}|"
        f"{chunk.page_content}"
    )

    chunk_id = hashlib.sha256(
        id_source.encode("utf-8")
    ).hexdigest()

    chunk_ids.append(chunk_id)


if len(set(chunk_ids)) != len(chunk_ids):
    raise ValueError(
        "Duplicate deterministic chunk IDs were generated."
    )


log_success(
    f"Created {len(chunk_ids)} unique deterministic IDs."
)


print("First chunk ID:", chunk_ids[0])
print("First chunk metadata:")
print(chunks[0].metadata)


SUCCESS: Created 150 unique deterministic IDs.
First chunk ID: 6418b5fb8e6c7f5959c2c53974ca0afa780f5848bbec6cb7221314f31907cc7d
First chunk metadata:
{'source': 'https://docs.langchain.com/oss/python/langchain/overview', 'source_domain': 'docs.langchain.com', 'source_path': '/oss/python/langchain/overview', 'crawler': 'tavily', 'content_format': 'markdown', 'page_content_hash': 'b7721a4893bc5070440abdd6b25e93bdfe5de1c505e698f4dfa8587546a5ad28', 'global_chunk_number': 0, 'source_chunk_number': 0, 'chunk_character_count': 3959, 'chunk_size_setting': 4000, 'chunk_overlap_setting': 200, 'embedding_model': 'gemini-embedding-001', 'embedding_dimension': 1536, 'ingestion_timestamp_utc': '2026-09-28T13:30:07.698404+00:00'}


## 16. Run metadata privacy and integrity checks

This prevents local paths, secret names, HTML anchors, and unsupported metadata values from being uploaded.


In [35]:
BLOCKED_METADATA_TERMS = [
    "onedrive",
    r"c:\\users",
    "appdata",
    "tavily_api_key",
    "pinecone_api_key",
    "google_api_key",
    "<a ",
    "href=",
]


chunk_metadata_issues = []


for chunk_number, chunk in enumerate(chunks):
    metadata_text = json.dumps(
        chunk.metadata,
        ensure_ascii=False,
    ).lower()

    detected_terms = [
        term
        for term in BLOCKED_METADATA_TERMS
        if term.lower() in metadata_text
    ]

    if detected_terms:
        chunk_metadata_issues.append({
            "chunk_number": chunk_number,
            "detected_terms": detected_terms,
        })

    unsupported_keys = [
        key
        for key, value in chunk.metadata.items()
        if not is_supported_metadata_value(value)
    ]

    if unsupported_keys:
        chunk_metadata_issues.append({
            "chunk_number": chunk_number,
            "unsupported_keys": unsupported_keys,
        })

    if not chunk.metadata["source"].startswith("https://"):
        chunk_metadata_issues.append({
            "chunk_number": chunk_number,
            "reason": "invalid_source_url",
        })


if chunk_metadata_issues:
    raise ValueError(
        f"Chunk metadata validation failed: "
        f"{chunk_metadata_issues[:10]}"
    )


log_success(
    "Chunk metadata privacy and integrity checks passed."
)


SUCCESS: Chunk metadata privacy and integrity checks passed.


## 17. Connect to and validate the existing Pinecone index

> **This cell makes Pinecone control-plane API requests.**

The expected index configuration is:

```text
Dimension: 1536
Metric: cosine
Serverless: AWS us-east-1
```


In [36]:
pinecone_client = Pinecone(
    api_key=os.environ["PINECONE_API_KEY"]
)


existing_index_names = [
    index_info["name"]
    for index_info in pinecone_client.list_indexes()
]


if INDEX_NAME not in existing_index_names:
    raise ValueError(
        f"Pinecone index '{INDEX_NAME}' does not exist."
    )


index_description = pinecone_client.describe_index(
    INDEX_NAME
)


print("Index name:", index_description.name)
print("Dimension:", index_description.dimension)
print("Metric:", index_description.metric)
print("Ready:", index_description.status["ready"])
print("Host:", index_description.host)


if index_description.dimension != EMBEDDING_DIMENSION:
    raise ValueError(
        f"Pinecone dimension is {index_description.dimension}, "
        f"but Gemini is configured for {EMBEDDING_DIMENSION}."
    )


if index_description.metric != "cosine":
    raise ValueError(
        f"Expected cosine metric, found {index_description.metric}."
    )


if not index_description.status["ready"]:
    raise RuntimeError(
        f"Pinecone index '{INDEX_NAME}' is not ready."
    )


Index name: langchain-docs-helper-gemini
Dimension: 1536
Metric: cosine
Ready: True
Host: langchain-docs-helper-gemini-upeditj.svc.aped-4627-b74a.pinecone.io


## 18. Check the namespace before ingestion

> **This cell makes one Pinecone data-plane request.**

A brand-new namespace should have zero records or be absent from the statistics.


In [17]:
pinecone_index = pinecone_client.Index(
    host=index_description.host
)


stats_before = pinecone_index.describe_index_stats()
namespace_before = stats_before.namespaces.get(
    NAMESPACE
)


vectors_before = (
    namespace_before.vector_count
    if namespace_before is not None
    else 0
)


print("Total index vectors before ingestion:", stats_before.total_vector_count)
print(
    f"Vectors in namespace '{NAMESPACE}' before ingestion:",
    vectors_before,
)


Total index vectors before ingestion: 0
Vectors in namespace 'langchain-docs' before ingestion: 0


## 19. Initialize Gemini embeddings

Creating the client object makes no embedding request.

The explicit output dimensionality must match the Pinecone index.


In [37]:
embeddings = GoogleGenerativeAIEmbeddings(
    model=EMBEDDING_MODEL,
    google_api_key=os.environ["GOOGLE_API_KEY"],
    output_dimensionality=EMBEDDING_DIMENSION,
)


print("Embedding client:", type(embeddings).__name__)
print("Model:", EMBEDDING_MODEL)
print("Requested dimension:", EMBEDDING_DIMENSION)


Embedding client: GoogleGenerativeAIEmbeddings
Model: gemini-embedding-001
Requested dimension: 1536


## 20. Test one embedding before processing all chunks

> **This cell makes one Gemini embedding request.**

Run it once to confirm credentials and dimensionality.


In [20]:
RUN_EMBEDDING_DIMENSION_TEST = True


if RUN_EMBEDDING_DIMENSION_TEST:
    test_vector = embeddings.embed_query(
        "Validate the documentation helper embedding dimension."
    )

    print("Returned embedding dimension:", len(test_vector))
    print("First five values:", test_vector[:5])

    if len(test_vector) != EMBEDDING_DIMENSION:
        raise ValueError(
            f"Expected {EMBEDDING_DIMENSION} values, "
            f"received {len(test_vector)}."
        )

    log_success("Gemini embedding dimension test passed.")
else:
    log_warning(
        "Dimension test not run. Set "
        "RUN_EMBEDDING_DIMENSION_TEST=True when ready."
    )


Returned embedding dimension: 1536
First five values: [-0.0017216926, 0.0010253235, -0.00361437, -0.08615047, 0.013893591]
SUCCESS: Gemini embedding dimension test passed.


## 21. Create the Pinecone vector-store connection

The namespace is specified explicitly so the documentation corpus remains isolated within the new index.


In [38]:
vector_store = PineconeVectorStore(
    index=pinecone_index,
    embedding=embeddings,
    namespace=NAMESPACE,
)


print("Vector store:", type(vector_store).__name__)
print("Namespace:", NAMESPACE)


Vector store: PineconeVectorStore
Namespace: langchain-docs


## 22. Preview the batching plan

The instructor batches documents before indexing. This notebook uses a conservative default batch size of 20 and processes batches sequentially.

Why sequential by default:

- The Gemini integration may issue one embedding API call per chunk.
- Concurrent batches can exceed requests-per-minute limits quickly.
- Sequential batches make errors easier to diagnose.
- This is a learning workload, not a throughput benchmark.


In [39]:
batches = [
    (
        chunks[start_index:start_index + BATCH_SIZE],
        chunk_ids[start_index:start_index + BATCH_SIZE],
    )
    for start_index in range(
        0,
        len(chunks),
        BATCH_SIZE,
    )
]


print("Total chunks:", len(chunks))
print("Batch size:", BATCH_SIZE)
print("Number of batches:", len(batches))


for batch_number, (batch_docs, batch_ids) in enumerate(
    batches,
    start=1,
):
    print(
        f"Batch {batch_number:3d}: "
        f"{len(batch_docs):3d} documents, "
        f"{len(batch_ids):3d} IDs"
    )


Total chunks: 150
Batch size: 10
Number of batches: 15
Batch   1:  10 documents,  10 IDs
Batch   2:  10 documents,  10 IDs
Batch   3:  10 documents,  10 IDs
Batch   4:  10 documents,  10 IDs
Batch   5:  10 documents,  10 IDs
Batch   6:  10 documents,  10 IDs
Batch   7:  10 documents,  10 IDs
Batch   8:  10 documents,  10 IDs
Batch   9:  10 documents,  10 IDs
Batch  10:  10 documents,  10 IDs
Batch  11:  10 documents,  10 IDs
Batch  12:  10 documents,  10 IDs
Batch  13:  10 documents,  10 IDs
Batch  14:  10 documents,  10 IDs
Batch  15:  10 documents,  10 IDs


## 23. Define controlled sequential batch ingestion

Each batch records:

- Batch number
- Chunk count
- Status
- Duration
- Uploaded IDs or error message

If one batch fails, later batches can continue. The final summary identifies failed batches for targeted retry.


In [40]:
def index_documents_in_batches(
    vector_store,
    document_batches,
    pause_seconds=2,
):
    """
    Embed and upsert document batches sequentially.

    A batch still may result in multiple Gemini provider calls because
    the embedding integration can process documents individually.
    """

    log_header("VECTOR STORAGE PHASE")
    log_info(
        f"Preparing to add {sum(len(batch[0]) for batch in document_batches)} "
        f"chunks across {len(document_batches)} batches."
    )

    batch_results = []

    for batch_number, (batch_docs, batch_ids) in enumerate(
        document_batches,
        start=1,
    ):
        batch_start_time = time.perf_counter()

        log_info(
            f"Processing batch {batch_number}/{len(document_batches)} "
            f"with {len(batch_docs)} chunks."
        )

        try:
            uploaded_ids = vector_store.add_documents(
                documents=batch_docs,
                ids=batch_ids,
            )

            duration_seconds = (
                time.perf_counter() - batch_start_time
            )

            batch_results.append({
                "batch_number": batch_number,
                "status": "success",
                "chunk_count": len(batch_docs),
                "uploaded_id_count": len(uploaded_ids),
                "duration_seconds": round(duration_seconds, 2),
            })

            log_success(
                f"Batch {batch_number}/{len(document_batches)} uploaded "
                f"successfully in {duration_seconds:.2f} seconds."
            )

        except Exception as error:
            duration_seconds = (
                time.perf_counter() - batch_start_time
            )

            batch_results.append({
                "batch_number": batch_number,
                "status": "failed",
                "chunk_count": len(batch_docs),
                "duration_seconds": round(duration_seconds, 2),
                "error_type": type(error).__name__,
                "error_message": str(error),
            })

            log_error(
                f"Batch {batch_number}/{len(document_batches)} failed: "
                f"{type(error).__name__}: {error}"
            )

        if (
            batch_number < len(document_batches)
            and pause_seconds > 0
        ):
            log_info(
                f"Waiting {pause_seconds} seconds before the next batch."
            )
            time.sleep(pause_seconds)

    return batch_results


## 24. Run the controlled ingestion

> **This cell generates Gemini embeddings for every chunk and writes records to Pinecone.**

Before setting the flag to `True`:

1. Inspect the chunk previews.
2. Confirm metadata checks passed.
3. Confirm the Pinecone index is ready.
4. Confirm Gemini limits in Google AI Studio.
5. Ensure the namespace is correct.

Run this cell once.


In [41]:
RUN_DOCUMENTATION_INGESTION = True


if RUN_DOCUMENTATION_INGESTION:
    ingestion_started_at = datetime.now(
        timezone.utc
    ).isoformat()

    ingestion_start_time = time.perf_counter()

    batch_results = index_documents_in_batches(
        vector_store=vector_store,
        document_batches=batches,
        pause_seconds=SECONDS_BETWEEN_BATCHES,
    )

    ingestion_duration_seconds = round(
        time.perf_counter() - ingestion_start_time,
        2,
    )

    ingestion_finished_at = datetime.now(
        timezone.utc
    ).isoformat()

    log_success(
        f"Controlled ingestion run finished in "
        f"{ingestion_duration_seconds} seconds."
    )
else:
    batch_results = []
    ingestion_started_at = None
    ingestion_finished_at = None
    ingestion_duration_seconds = None

    log_warning(
        "Documentation ingestion not executed. Set "
        "RUN_DOCUMENTATION_INGESTION=True when ready."
    )



VECTOR STORAGE PHASE

INFO: Preparing to add 150 chunks across 15 batches.
INFO: Processing batch 1/15 with 10 chunks.
SUCCESS: Batch 1/15 uploaded successfully in 6.41 seconds.
INFO: Waiting 60 seconds before the next batch.
INFO: Processing batch 2/15 with 10 chunks.
SUCCESS: Batch 2/15 uploaded successfully in 5.75 seconds.
INFO: Waiting 60 seconds before the next batch.
INFO: Processing batch 3/15 with 10 chunks.
SUCCESS: Batch 3/15 uploaded successfully in 5.27 seconds.
INFO: Waiting 60 seconds before the next batch.
INFO: Processing batch 4/15 with 10 chunks.
SUCCESS: Batch 4/15 uploaded successfully in 6.13 seconds.
INFO: Waiting 60 seconds before the next batch.
INFO: Processing batch 5/15 with 10 chunks.
SUCCESS: Batch 5/15 uploaded successfully in 5.04 seconds.
INFO: Waiting 60 seconds before the next batch.
INFO: Processing batch 6/15 with 10 chunks.
SUCCESS: Batch 6/15 uploaded successfully in 5.18 seconds.
INFO: Waiting 60 seconds before the next batch.
INFO: Processing b

## 25. Summarize batch outcomes

This cell makes no API calls.


In [42]:
if not batch_results:
    log_warning("No batch results are available yet.")
else:
    successful_batches = [
        result
        for result in batch_results
        if result["status"] == "success"
    ]

    failed_batches = [
        result
        for result in batch_results
        if result["status"] == "failed"
    ]

    successful_chunks = sum(
        result["chunk_count"]
        for result in successful_batches
    )

    failed_chunks = sum(
        result["chunk_count"]
        for result in failed_batches
    )

    log_header("BATCH INGESTION SUMMARY")
    print("Total batches:", len(batch_results))
    print("Successful batches:", len(successful_batches))
    print("Failed batches:", len(failed_batches))
    print("Successful chunks:", successful_chunks)
    print("Failed chunks:", failed_chunks)

    if failed_batches:
        print()
        print("Failed batch details:")

        for result in failed_batches:
            print(result)



BATCH INGESTION SUMMARY

Total batches: 15
Successful batches: 15
Failed batches: 0
Successful chunks: 150
Failed chunks: 0


## 26. Inspect Pinecone statistics after ingestion

> **This cell makes one Pinecone data-plane request.**

Pinecone can be eventually consistent, so statistics may take a short time to reflect the latest upserts.


In [43]:
stats_after = pinecone_index.describe_index_stats()
namespace_after = stats_after.namespaces.get(
    NAMESPACE
)


vectors_after = (
    namespace_after.vector_count
    if namespace_after is not None
    else 0
)


print("Total index vectors after ingestion:", stats_after.total_vector_count)
print(
    f"Vectors in namespace '{NAMESPACE}' after ingestion:",
    vectors_after,
)
print(
    "Namespace vector-count change:",
    vectors_after - vectors_before,
)
print("Expected deterministic chunk count:", len(chunk_ids))


Total index vectors after ingestion: 150
Vectors in namespace 'langchain-docs' after ingestion: 150
Namespace vector-count change: 150
Expected deterministic chunk count: 150


## 27. Save an ingestion manifest

The manifest records configuration, counts, and batch outcomes without storing secret values.


In [44]:
if batch_results:
    ingestion_manifest = {
        "index_name": INDEX_NAME,
        "namespace": NAMESPACE,
        "embedding_model": EMBEDDING_MODEL,
        "embedding_dimension": EMBEDDING_DIMENSION,
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
        "source_document_count": len(source_documents),
        "chunk_count": len(chunks),
        "batch_size": BATCH_SIZE,
        "batch_count": len(batches),
        "ingestion_started_at_utc": ingestion_started_at,
        "ingestion_finished_at_utc": ingestion_finished_at,
        "ingestion_duration_seconds": ingestion_duration_seconds,
        "vectors_before": vectors_before,
        "vectors_after": vectors_after,
        "batch_results": batch_results,
    }

    INGESTION_MANIFEST_PATH.write_text(
        json.dumps(
            ingestion_manifest,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    log_success(
        f"Saved ingestion manifest to: {INGESTION_MANIFEST_PATH}"
    )
else:
    log_warning("No ingestion manifest written because ingestion did not run.")


SUCCESS: Saved ingestion manifest to: C:\Users\inkk1r04\OneDrive - Kellogg Company\Desktop\Work\0. DS Capability\Courses\Langchain\langchain-course-jupyter\documentation_crawl_artifacts\langchain_docs_ingestion_manifest.json


## 28. Run a retrieval sanity check

> **This cell makes one Gemini query-embedding request and one Pinecone query request. It does not call a chat model.**

The goal is to verify that documentation chunks can be retrieved by meaning before building the backend in notebook 12.


In [46]:
RUN_RETRIEVAL_SANITY_CHECK = False


if RUN_RETRIEVAL_SANITY_CHECK:
    sanity_query = (
        "How do I create and invoke a LangChain runnable?"
    )

    sanity_results = vector_store.similarity_search_with_score(
        query=sanity_query,
        k=3,
    )

    print("Query:", sanity_query)
    print("Retrieved results:", len(sanity_results))

    for result_number, (document, score) in enumerate(
        sanity_results,
        start=1,
    ):
        print("=" * 100)
        print("Result:", result_number)
        print("Similarity score:", round(score, 4))
        print("Source:", document.metadata.get("source"))
        print(
            "Source chunk number:",
            document.metadata.get("source_chunk_number"),
        )
        print("Content preview:")
        print(document.page_content[:1400])
        print()
else:
    log_warning(
        "Retrieval sanity check not run. Set "
        "RUN_RETRIEVAL_SANITY_CHECK=True after ingestion."
    )


Query: How do I create and invoke a LangChain runnable?
Retrieved results: 3
Result: 1
Similarity score: 0.7242
Source: https://docs.langchain.com/oss/python/langchain/agents
Source chunk number: 2.0
Content preview:
class Answer(BaseModel):
 summary: str
 confidence: float


agent = create_agent(model="fireworks:accounts/fireworks/models/glm-5p2", tools=tools, response_format=Answer)
result = agent.invoke({"messages": [{"role": "user", "content": "Summarize AI trends"}]})
result["structured_response"] # Answer(summary=..., confidence=...)`
`from pydantic import BaseModel
from langchain.agents import create_agent


class Answer(BaseModel):
 summary: str
 confidence: float


agent = create_agent(model="baseten:zai-org/GLM-5.2", tools=tools, response_format=Answer)
result = agent.invoke({"messages": [{"role": "user", "content": "Summarize AI trends"}]})
result["structured_response"] # Answer(summary=..., confidence=...)`
`from pydantic import BaseModel
from langchain.agents import create

# Optional: controlled asynchronous batching

The instructor uses `aadd_documents()` and `asyncio.gather()` to process batches asynchronously.

The following implementation preserves that concept while limiting concurrent batches with a semaphore. It is disabled because concurrent Gemini calls can reach free-tier RPM limits quickly.

Do not run the asynchronous ingestion after the sequential ingestion merely as a demonstration. It would regenerate every embedding and upsert every record again.


In [ ]:
async def index_documents_async_controlled(
    vector_store,
    document_batches,
    max_concurrent_batches=2,
):
    """Index batches asynchronously with bounded concurrency."""

    semaphore = asyncio.Semaphore(
        max_concurrent_batches
    )

    async def add_batch(
        batch_number,
        batch_docs,
        batch_ids,
    ):
        async with semaphore:
            batch_start_time = time.perf_counter()

            try:
                uploaded_ids = await vector_store.aadd_documents(
                    documents=batch_docs,
                    ids=batch_ids,
                )

                duration_seconds = round(
                    time.perf_counter() - batch_start_time,
                    2,
                )

                log_success(
                    f"Async batch {batch_number}/"
                    f"{len(document_batches)} uploaded successfully."
                )

                return {
                    "batch_number": batch_number,
                    "status": "success",
                    "chunk_count": len(batch_docs),
                    "uploaded_id_count": len(uploaded_ids),
                    "duration_seconds": duration_seconds,
                }

            except Exception as error:
                duration_seconds = round(
                    time.perf_counter() - batch_start_time,
                    2,
                )

                log_error(
                    f"Async batch {batch_number} failed: {error}"
                )

                return {
                    "batch_number": batch_number,
                    "status": "failed",
                    "chunk_count": len(batch_docs),
                    "duration_seconds": duration_seconds,
                    "error_type": type(error).__name__,
                    "error_message": str(error),
                }

    tasks = [
        add_batch(
            batch_number,
            batch_docs,
            batch_ids,
        )
        for batch_number, (batch_docs, batch_ids) in enumerate(
            document_batches,
            start=1,
        )
    ]

    return await asyncio.gather(*tasks)


print(
    "Controlled async ingestion function defined but not executed."
)


## 29. Optional namespace cleanup

Use this only when intentionally rebuilding the entire documentation corpus.

> **Warning:** This permanently deletes every record in the `langchain-docs` namespace.

A rebuild is appropriate when:

- The crawl source changes substantially
- The chunking strategy changes
- The embedding model changes
- The embedding dimensionality changes
- Old records must be removed after pages disappear from the crawl


In [ ]:
# DELETE_DOCUMENTATION_NAMESPACE = False
#
#
# if DELETE_DOCUMENTATION_NAMESPACE:
#     pinecone_index.delete(
#         delete_all=True,
#         namespace=NAMESPACE,
#     )
#
#     log_warning(
#         f"Deleted all records from namespace '{NAMESPACE}'."
#     )


## 30. API usage expectations

For `N` documentation chunks, the ingestion integration may generate approximately:

```text
N Gemini document-embedding requests
+ Pinecone upsert request(s), grouped by batch
```

Your actual Gemini dashboard is the best indicator of provider-level request behavior.

The following actions do not call Gemini:

- Loading JSONL
- Splitting text
- Creating deterministic IDs
- Metadata checks
- Constructing the embedding client object

The following actions call Gemini:

- Dimension test: one request
- Ingestion: approximately one request per chunk in the observed integration behavior
- Retrieval sanity check: one query-embedding request

The following actions call Pinecone:

- Index validation
- Namespace statistics
- Batch upserts
- Retrieval sanity query


## 31. Handoff to notebook 12

After successful ingestion and retrieval validation, notebook 12 will implement the reusable documentation RAG backend:

```text
User question
        ↓
Gemini query embedding
        ↓
Pinecone retrieval from langchain-docs
        ↓
Retrieved documentation chunks and sources
        ↓
Grounded prompt
        ↓
Gemini chat model
        ↓
{
    "answer": ...,
    "context": [Document, Document, ...]
}
```

Notebook 12 will preserve the role of the instructor's `backend/core.py`, including a reusable `run_llm()` function and source-citation support for the later Streamlit application.
